# CNS2025 Homework 14

Rectifying recurrent network with cosine connectivity.

_This notebook contains my solutions for Exercises 1 and 2._

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Use a simple style for plots
plt.rcParams.update({
    "figure.figsize": (6, 4),
    "axes.spines.top": False,
    "axes.spines.right": False
})

# Angle grid and connectivity kernel (shared by all exercises)
dth = 0.1
ths = np.arange(-np.pi, np.pi, dth)  # angles in radians
lbd1 = 1.9

# Cosine connectivity kernel M_{ij} = cos(theta_i - theta_j)
M = np.cos(ths[:, None] - ths[None, :])


In [ ]:

def run_recurrent(h, lbd1=1.9, dth=0.1, dt=0.01, tol=1e-8, max_iter=100000):
    """Iterate the rectifying recurrent network to steady state.

    tau_r is set to 1, so the dynamics are:
        dv/dt = -v + [h + (lbd1/pi) * ∫ cos(theta - theta') v(theta') dtheta']_+
    """
    v = np.zeros_like(h)
    err = np.inf
    itn = 0

    while err > tol and itn < max_iter:
        x = h + (lbd1 * dth / np.pi) * (M @ v)
        x = np.maximum(x, 0.0)
        dv = x - v
        v = v + dt * dv
        err = np.linalg.norm(dv)
        itn += 1

    print(f"converged in {itn} steps, err = {err:.2e}")
    return v


def fourier_amplitudes(x, max_mode=9):
    """Compute |Fourier coefficient| for modes 0..max_mode on the circle."""
    amps = []
    for mu in range(max_mode + 1):
        c = np.sum(np.cos(mu * ths) * x) * dth / np.pi
        s = np.sum(np.sin(mu * ths) * x) * dth / np.pi
        amps.append(np.sqrt(c**2 + s**2))
    return np.array(amps)


In [ ]:

# Exercise 1: two-bump input and winner-takes-all behavior

# Generate input h(theta) as specified
h0 = np.cos(ths + np.pi / 2.0) - 0.8
h0[h0 < 0] = 0.0

h1 = np.cos(ths - np.pi / 2.0) - 0.82
h1[h1 < 0] = 0.0

h = h0 + h1

# Smoothing filter
k = np.exp(-np.linspace(-10, 10, int(2 // dth))**2 / 10.0)
k /= np.sum(k)
h = np.convolve(h, k, mode="same")

# Run recurrent network
v = run_recurrent(h, lbd1=lbd1, dth=dth)

# Angles in degrees for plotting
deg = ths * 180.0 / np.pi

# Plot input/output and their Fourier amplitudes (similar to Slide 20)
fig, axes = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)

axes[0, 0].plot(deg, h)
axes[0, 0].set_title("Input $h(\\theta)$")
axes[0, 0].set_xlabel("$\\theta$ (deg)")
axes[0, 0].set_ylabel("h")

axes[0, 1].plot(deg, v)
axes[0, 1].set_title("Output $v(\\theta)$")
axes[0, 1].set_xlabel("$\\theta$ (deg)")
axes[0, 1].set_ylabel("v")

modes = np.arange(10)
amp_h = fourier_amplitudes(h, max_mode=9)
amp_v = fourier_amplitudes(v, max_mode=9)

axes[1, 0].bar(modes, amp_h)
axes[1, 0].set_title("Fourier amplitudes of input")
axes[1, 0].set_xlabel("mode $\\mu$")
axes[1, 0].set_ylabel("|c$_{\\mu}$|")

axes[1, 1].bar(modes, amp_v)
axes[1, 1].set_title("Fourier amplitudes of output")
axes[1, 1].set_xlabel("mode $\\mu$")
axes[1, 1].set_ylabel("|c$_{\\mu}$|")

plt.show()


In [ ]:

# Exercise 2: gain modulation with different offset levels

def make_input_with_offset(level, A=40.0, eps=0.1):
    """Construct an input with the same shape but different offset (contrast-like level)."""
    # Shape is (1 - eps) + eps * cos(2 theta); level scales both baseline and peak
    return A * level * (1.0 - eps + eps * np.cos(2.0 * ths))

levels = [0.5, 1.0, 1.5]  # three different offset/contrast levels
inputs = [make_input_with_offset(c) for c in levels]
outputs = [run_recurrent(h_in, lbd1=lbd1, dth=dth) for h_in in inputs]

deg = ths * 180.0 / np.pi

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

# Plot A: three inputs
for h_in, c in zip(inputs, levels):
    axes[0].plot(deg, h_in, label=f"level = {c}")
axes[0].set_title("Exercise 2: Inputs (Plot A)")
axes[0].set_xlabel("$\\theta$ (deg)")
axes[0].set_ylabel("h($\\theta$)")
axes[0].legend()

# Plot B: corresponding outputs
for v_out, c in zip(outputs, levels):
    axes[1].plot(deg, v_out, label=f"level = {c}")
axes[1].set_title("Exercise 2: Outputs (Plot B)")
axes[1].set_xlabel("$\\theta$ (deg)")
axes[1].set_ylabel("v($\\theta$)")
axes[1].legend()

plt.show()

# Quick check of gain modulation: print peak outputs
for c, v_out in zip(levels, outputs):
    print(f"level {c}: max v(theta) = {v_out.max():.3f}")
